# CTGAN Synthetic Data Generation for Class Imbalance

## Overview
Generate synthetic click examples using CTGAN to address class imbalance (98.5% no-clicks vs 1.5% clicks).

**Hybrid Approach**: 
- Work with raw data before encoding
- Apply CTGAN-specific preprocessing (target encode very high-cardinality features)
- Generate synthetic clicks to achieve 85-15 ratio
- Apply standard encoding pipeline before modeling

**Data Flow**:
1. Load and merge raw data from CSV files
2. Target encode very high-cardinality features (1000+ unique values)
3. Group rare categories for medium-cardinality features (100-1000 unique)
4. Prepare data for CTGAN (hybrid preprocessing)
5. Train CTGAN on minority class (clicks only)
6. Generate synthetic clicks to achieve 85-15 ratio
7. Combine real + synthetic data
8. Apply standard encoding pipeline
9. Train and evaluate logistic regression
10. Compare against baseline models

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, confusion_matrix, 
    RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.model_selection import train_test_split

# Set random seed for reproducibility
SEED = 2025
np.random.seed(SEED)

# Display settings
%matplotlib inline
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully")

## 2. Load and Merge Raw Data

Load original CSV files and perform same merging process as `data_preprocessing.py`:
- Load user-level data (`train_data_ads.csv`) and interaction data (`train_data_feeds.csv`)
- Create user aggregation features from feeds data
- Merge user aggregates back to user data
- Verify merged data structure and class distribution

In [ ]:
def to01(s):
    """Convert label from -1/1 to 0/1"""
    s = s.copy()
    s = s.replace({-1: 0, 1: 1})
    return s.astype(int)

def safe_keep(df, cols, name):
    """Safely keep columns, reporting missing ones"""
    keep = [c for c in cols if c in df.columns]
    miss = [c for c in cols if c not in df.columns]
    if miss:
        print(f"[{name}] missing in the dataset: {miss}")
    return keep

print("Loading data...")
train_user = pd.read_csv('train_data_ads.csv')
train_adv = pd.read_csv('train_data_feeds.csv')

print(f"User data shape: {train_user.shape}")
print(f"Feeds data shape: {train_adv.shape}")

# Rename user_id column in feeds data
train_adv.rename(columns={'u_userId': 'user_id'}, inplace=True)

# Define variables to keep - using all available features for CTGAN
user_var = [
    'user_id', 'log_id', 'label',
    'age', 'gender', 'residence', 'city',
    'device_name', 'device_size', 'net_type',
    'task_id', 'adv_id', 'adv_prim_id', 'creat_type_cd',
    'city_rank', 'slot_id', 'spread_app_id',
    'inter_type_cd', 'series_group', 'series_dev', 'emui_dev',
    'hispace_app_tags', 'app_second_class', 'site_id'
]

adv_var = ['user_id', 'label']

# Safe keep columns
train_user_cols = safe_keep(train_user, user_var, "train_user")
train_adv_cols = safe_keep(train_adv, adv_var, "train_adv")

train_user = train_user[train_user_cols]
train_adv = train_adv[train_adv_cols]

print("\nData loaded successfully")

In [ ]:
# Create user aggregation features from feeds data
print("Creating user aggregation features...")

train_adv['label01'] = to01(train_adv['label'])

user_agg = (
    train_adv
    .groupby('user_id', as_index=False)
    .agg(
        feeds_imps=('label01', 'count'),
        feeds_clicks=('label01', 'sum'),
        feeds_ctr=('label01', 'mean')
    )
)

print(f"User aggregates shape: {user_agg.shape}")
print("\nSample aggregates:")
print(user_agg.head())

In [ ]:
# Merge user data with aggregates
print("Merging user data with aggregates...")
train_merged = train_user.merge(user_agg, on='user_id', how='left')

# Fill missing values
for c in ['feeds_imps', 'feeds_clicks', 'feeds_ctr']:
    if c in train_merged.columns:
        fill_val = 0 if c != 'feeds_ctr' else train_merged[c].mean()
        train_merged[c] = train_merged[c].fillna(fill_val)

# Convert label to 0/1
train_merged['label'] = to01(train_merged['label'])

print(f"\nMerged data shape: {train_merged.shape}")
print(f"Columns: {list(train_merged.columns)}")

# Clean up memory
del train_user, train_adv, user_agg
gc.collect()

print("\nMerge complete!")

## 3. Verify Data Structure and Class Distribution

In [ ]:
# Check data info
print("=" * 80)
print("MERGED DATA SUMMARY")
print("=" * 80)
print(f"\nShape: {train_merged.shape}")
print(f"\nData types:")
print(train_merged.dtypes)

print("\n" + "=" * 80)
print("CLASS DISTRIBUTION")
print("=" * 80)
class_dist = train_merged['label'].value_counts().sort_index()
print(f"\nNo-clicks (0): {class_dist[0]:,} ({class_dist[0]/len(train_merged)*100:.2f}%)")
print(f"Clicks (1): {class_dist[1]:,} ({class_dist[1]/len(train_merged)*100:.2f}%)")
print(f"\nImbalance ratio: {class_dist[0]/class_dist[1]:.2f}:1")

# Visualize class distribution
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
class_dist.plot(kind='bar', ax=ax[0], color=['#2E86AB', '#A23B72'])
ax[0].set_title('Class Distribution (Counts)', fontsize=14, fontweight='bold')
ax[0].set_xlabel('Label (0 = No Click, 1 = Click)')
ax[0].set_ylabel('Count')
ax[0].set_xticklabels(['No Click', 'Click'], rotation=0)

# Percentage plot
(class_dist / len(train_merged) * 100).plot(kind='bar', ax=ax[1], color=['#2E86AB', '#A23B72'])
ax[1].set_title('Class Distribution (Percentage)', fontsize=14, fontweight='bold')
ax[1].set_xlabel('Label (0 = No Click, 1 = Click)')
ax[1].set_ylabel('Percentage (%)')
ax[1].set_xticklabels(['No Click', 'Click'], rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze feature cardinality
print("=" * 80)
print("FEATURE CARDINALITY ANALYSIS")
print("=" * 80)

cardinality = {}
for col in train_merged.columns:
    if col not in ['user_id', 'log_id', 'label']:
        n_unique = train_merged[col].nunique()
        cardinality[col] = n_unique

cardinality_df = pd.DataFrame.from_dict(
    cardinality, orient='index', columns=['Unique Values']
).sort_values('Unique Values', ascending=False)

# Categorize by cardinality
cardinality_df['Category'] = pd.cut(
    cardinality_df['Unique Values'],
    bins=[-1, 10, 100, 1000, np.inf],
    labels=['Low (≤10)', 'Medium (11-100)', 'High (101-1000)', 'Very High (1000+)']
)

print("\nCardinality by feature:")
print(cardinality_df)

print("\n" + "=" * 80)
print("ENCODING STRATEGY")
print("=" * 80)
print("\nLow cardinality (≤10) - Keep as categorical for CTGAN:")
low_card = cardinality_df[cardinality_df['Category'] == 'Low (≤10)'].index.tolist()
for feat in low_card:
    print(f"  - {feat} ({cardinality_df.loc[feat, 'Unique Values']} unique)")

print("\nMedium cardinality (11-100) - Keep as categorical, may group rare:")
med_card = cardinality_df[cardinality_df['Category'] == 'Medium (11-100)'].index.tolist()
for feat in med_card:
    print(f"  - {feat} ({cardinality_df.loc[feat, 'Unique Values']} unique)")

print("\nHigh cardinality (101-1000) - Group rare categories:")
high_card = cardinality_df[cardinality_df['Category'] == 'High (101-1000)'].index.tolist()
for feat in high_card:
    print(f"  - {feat} ({cardinality_df.loc[feat, 'Unique Values']} unique)")

print("\nVery high cardinality (1000+) - Target encode to continuous:")
very_high_card = cardinality_df[cardinality_df['Category'] == 'Very High (1000+)'].index.tolist()
for feat in very_high_card:
    print(f"  - {feat} ({cardinality_df.loc[feat, 'Unique Values']} unique)")

## 4. Target Encode Very High-Cardinality Features

**Goal**: Convert very high-cardinality features (1000+ unique values) to continuous features via target encoding BEFORE CTGAN training.

**Rationale**: CTGAN struggles with 10K+ categories. Target encoding converts these to continuous numeric features that CTGAN can model effectively.

**Features to encode**:
- `task_id` (11,209 unique)
- `adv_id` (12,615 unique) - if present
- `device_size` (1,547 unique)

In [ ]:
def target_encode_for_ctgan(df, column, target='label'):
    """
    Apply target encoding to a high-cardinality feature.
    
    Args:
        df: DataFrame containing the data
        column: Column name to encode
        target: Target column name (default: 'label')
    
    Returns:
        df: DataFrame with new encoded column and original column removed
        encoding_map: Dictionary mapping original values to encoded values
    """
    # Compute mean target value for each category
    encoding_map = df.groupby(column)[target].mean().to_dict()
    
    # Create new encoded column
    encoded_col_name = f"{column}_encoded"
    df[encoded_col_name] = df[column].map(encoding_map)
    
    # Fill missing values with global mean
    global_mean = df[target].mean()
    df[encoded_col_name] = df[encoded_col_name].fillna(global_mean)
    
    # Drop original column
    df = df.drop(columns=[column])
    
    return df, encoding_map

print("Target encoding function defined")

In [ ]:
# Apply target encoding to very high-cardinality features
print("Applying target encoding to very high-cardinality features...\n")

# Store encoding maps for reference
encoding_maps = {}

# Features to encode (1000+ unique values)
very_high_card_features = ['task_id', 'device_size']

# Add adv_id if present (might be dropped in preprocessing)
if 'adv_id' in train_merged.columns:
    very_high_card_features.append('adv_id')

train_processed = train_merged.copy()

for feat in very_high_card_features:
    if feat in train_processed.columns:
        print(f"Encoding {feat}...")
        n_unique_before = train_processed[feat].nunique()
        
        train_processed, encoding_map = target_encode_for_ctgan(
            train_processed, feat, target='label'
        )
        
        encoding_maps[feat] = encoding_map
        
        encoded_col = f"{feat}_encoded"
        print(f"  Original unique values: {n_unique_before}")
        print(f"  Encoded column: {encoded_col}")
        print(f"  Encoded range: [{train_processed[encoded_col].min():.6f}, {train_processed[encoded_col].max():.6f}]")
        print(f"  Encoded mean: {train_processed[encoded_col].mean():.6f}\n")

print("Target encoding complete!")
print(f"\nProcessed data shape: {train_processed.shape}")

## 5. Group Rare Categories for Medium-High Cardinality Features

**Goal**: Reduce cardinality of features with 100-1000 unique values by grouping rare categories.

**Strategy**:
- `adv_prim_id` (545 unique) → Group to top 100, rest → "other"
- `city` (341 unique) → Group to top 50-100, rest → "other"
- `device_name` (256 unique) → Keep as is (manageable for CTGAN)

In [ ]:
def group_rare_categories(df, column, top_n, other_label='other'):
    """
    Group rare categories by keeping only top N most frequent categories.
    
    Args:
        df: DataFrame containing the data
        column: Column name to process
        top_n: Number of top categories to keep
        other_label: Label for grouped rare categories (default: 'other')
    
    Returns:
        df: DataFrame with grouped categories
        mapping: Dictionary mapping original to grouped values
    """
    # Count frequency of each category
    value_counts = df[column].value_counts()
    
    # Get top N categories
    top_categories = value_counts.head(top_n).index.tolist()
    
    # Create mapping
    mapping = {cat: cat if cat in top_categories else other_label 
               for cat in df[column].unique()}
    
    # Apply mapping
    df[column] = df[column].map(mapping)
    
    return df, mapping

print("Grouping function defined")

In [ ]:
# Apply grouping to high-cardinality features
print("Grouping rare categories for high-cardinality features...\n")

grouping_maps = {}

# Group adv_prim_id to top 100
if 'adv_prim_id' in train_processed.columns:
    print("Grouping adv_prim_id...")
    n_before = train_processed['adv_prim_id'].nunique()
    train_processed, mapping = group_rare_categories(
        train_processed, 'adv_prim_id', top_n=100, other_label='other'
    )
    grouping_maps['adv_prim_id'] = mapping
    n_after = train_processed['adv_prim_id'].nunique()
    print(f"  Before: {n_before} unique values")
    print(f"  After: {n_after} unique values\n")

# Group city to top 50
if 'city' in train_processed.columns:
    print("Grouping city...")
    n_before = train_processed['city'].nunique()
    train_processed, mapping = group_rare_categories(
        train_processed, 'city', top_n=50, other_label='other'
    )
    grouping_maps['city'] = mapping
    n_after = train_processed['city'].nunique()
    print(f"  Before: {n_before} unique values")
    print(f"  After: {n_after} unique values\n")

print("Grouping complete!")
print(f"\nProcessed data shape: {train_processed.shape}")

## 6. Prepare Data for CTGAN

**Goal**: Prepare data for CTGAN training by:
1. Separating clicks (minority class) from no-clicks
2. Dropping ID columns and target variable
3. Ensuring proper data types for CTGAN
4. Creating metadata for CTGAN

In [ ]:
# Separate clicks and no-clicks
print("Separating clicks and no-clicks...\n")

clicks_data = train_processed[train_processed['label'] == 1].copy()
no_clicks_data = train_processed[train_processed['label'] == 0].copy()

print(f"Clicks: {len(clicks_data):,} ({len(clicks_data)/len(train_processed)*100:.2f}%)")
print(f"No-clicks: {len(no_clicks_data):,} ({len(no_clicks_data)/len(train_processed)*100:.2f}%)")

print("\n" + "=" * 80)
print("TARGET RATIO CALCULATION")
print("=" * 80)
print(f"\nCurrent distribution:")
print(f"  No-clicks: {len(no_clicks_data):,}")
print(f"  Clicks: {len(clicks_data):,}")

# Calculate synthetic clicks needed for 85-15 ratio
n_no_clicks = len(no_clicks_data)
target_clicks = int((n_no_clicks / 0.85) * 0.15)
synthetic_clicks_needed = target_clicks - len(clicks_data)

print(f"\nTarget distribution (85-15 ratio):")
print(f"  No-clicks: {n_no_clicks:,} (85%)")
print(f"  Total clicks needed: {target_clicks:,} (15%)")
print(f"  Synthetic clicks to generate: {synthetic_clicks_needed:,}")
print(f"\nFinal dataset size: {n_no_clicks + target_clicks:,}")

In [ ]:
# Prepare CTGAN training data (clicks only)
print("Preparing data for CTGAN...\n")

# Drop ID columns and label
ctgan_data = clicks_data.drop(columns=['user_id', 'log_id', 'label'])

print(f"CTGAN training data shape: {ctgan_data.shape}")
print(f"\nFeatures for CTGAN: {list(ctgan_data.columns)}")

# Identify categorical and continuous columns
categorical_cols = []
continuous_cols = []

for col in ctgan_data.columns:
    if col.endswith('_encoded') or ctgan_data[col].dtype in ['float64', 'float32']:
        continuous_cols.append(col)
    else:
        categorical_cols.append(col)
        # Convert to string for CTGAN
        ctgan_data[col] = ctgan_data[col].astype(str)

print(f"\nCategorical features ({len(categorical_cols)}): {categorical_cols}")
print(f"\nContinuous features ({len(continuous_cols)}): {continuous_cols}")

## 7. Install and Train CTGAN

**Installation**: Install the CTGAN library from SDV (Synthetic Data Vault)

**Training**: Train CTGAN on minority class (clicks only) to learn the distribution

In [ ]:
# Install CTGAN
!pip install ctgan --quiet

from ctgan import CTGAN

print("CTGAN library installed and imported successfully")

In [ ]:
# Initialize and train CTGAN
print("Initializing CTGAN...\n")

ctgan = CTGAN(
    epochs=100,              # Number of training epochs (adjust based on performance)
    batch_size=500,          # Batch size for training
    generator_lr=2e-4,       # Learning rate for generator
    discriminator_lr=2e-4,   # Learning rate for discriminator
    verbose=True
)

print("Training CTGAN on click examples...")
print(f"Training samples: {len(ctgan_data):,}")
print(f"Training features: {len(ctgan_data.columns)}")
print(f"\nThis may take 10-30 minutes depending on data size...\n")

# Train CTGAN
ctgan.fit(ctgan_data, discrete_columns=categorical_cols)

print("\nCTGAN training complete!")

In [ ]:
# Save trained CTGAN model for reproducibility
import pickle

with open('ctgan_model.pkl', 'wb') as f:
    pickle.dump(ctgan, f)

print("CTGAN model saved to: ctgan_model.pkl")

## 8. Generate Synthetic Clicks

**Goal**: Generate synthetic click examples to achieve 85-15 ratio (no-clicks : clicks)

**Strategy**: Generate in batches to manage memory, then combine all synthetic samples

In [ ]:
# Generate synthetic clicks
print("Generating synthetic click samples...\n")
print(f"Synthetic samples needed: {synthetic_clicks_needed:,}")

# Generate in batches to manage memory
batch_size = 100000
n_batches = (synthetic_clicks_needed // batch_size) + 1
remaining = synthetic_clicks_needed

synthetic_samples = []

for i in range(n_batches):
    n_to_generate = min(batch_size, remaining)
    if n_to_generate > 0:
        print(f"Batch {i+1}/{n_batches}: Generating {n_to_generate:,} samples...")
        batch = ctgan.sample(n_to_generate)
        synthetic_samples.append(batch)
        remaining -= n_to_generate

# Combine all batches
synthetic_clicks = pd.concat(synthetic_samples, ignore_index=True)

print(f"\nSynthetic generation complete!")
print(f"Total synthetic samples: {len(synthetic_clicks):,}")
print(f"Synthetic data shape: {synthetic_clicks.shape}")

In [ ]:
# Inspect synthetic data
print("=" * 80)
print("SYNTHETIC DATA QUALITY CHECK")
print("=" * 80)

# Check for NaN values
nan_counts = synthetic_clicks.isna().sum()
if nan_counts.sum() > 0:
    print("\nWARNING: Found NaN values:")
    print(nan_counts[nan_counts > 0])
else:
    print("\n✓ No NaN values found")

# Check data types
print("\nData types:")
print(synthetic_clicks.dtypes)

# Sample rows
print("\nSample synthetic data:")
print(synthetic_clicks.head())

# Add synthetic flag
synthetic_clicks['is_synthetic'] = 1
print("\n✓ Added 'is_synthetic' flag to synthetic data")

## 9. Compare Real vs Synthetic Data Distributions

Validate synthetic data quality by comparing distributions of key features

In [ ]:
# Compare distributions of key features
print("Comparing real vs synthetic data distributions...\n")

# Select a few key features to compare
features_to_compare = [
    col for col in continuous_cols[:5]  # First 5 continuous features
] + [
    col for col in categorical_cols[:3]  # First 3 categorical features
]

n_features = len(features_to_compare)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
axes = axes.flatten() if n_features > 1 else [axes]

for idx, feat in enumerate(features_to_compare):
    ax = axes[idx]
    
    if feat in continuous_cols:
        # Histogram for continuous features
        ax.hist(ctgan_data[feat], bins=50, alpha=0.5, label='Real', density=True, color='blue')
        ax.hist(synthetic_clicks[feat], bins=50, alpha=0.5, label='Synthetic', density=True, color='red')
        ax.set_xlabel(feat)
        ax.set_ylabel('Density')
    else:
        # Bar plot for categorical features
        real_counts = ctgan_data[feat].value_counts(normalize=True).head(10)
        synth_counts = synthetic_clicks[feat].value_counts(normalize=True).head(10)
        
        x = np.arange(len(real_counts))
        width = 0.35
        
        ax.bar(x - width/2, real_counts.values, width, label='Real', alpha=0.8, color='blue')
        ax.bar(x + width/2, synth_counts.values, width, label='Synthetic', alpha=0.8, color='red')
        ax.set_xlabel(feat)
        ax.set_ylabel('Proportion')
        ax.set_xticks(x)
        ax.set_xticklabels(real_counts.index, rotation=45, ha='right')
    
    ax.set_title(f'{feat} Distribution', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Hide extra subplots
for idx in range(n_features, len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

print("Distribution comparison complete!")

## 10. Combine Real and Synthetic Data

Combine real no-clicks, real clicks, and synthetic clicks to create final balanced dataset

In [ ]:
# Prepare real data for combination
print("Preparing real data...\n")

# Add synthetic flag to real data
real_clicks = clicks_data.copy()
real_no_clicks = no_clicks_data.copy()

real_clicks['is_synthetic'] = 0
real_no_clicks['is_synthetic'] = 0

# Add label back to synthetic clicks
synthetic_clicks['label'] = 1

# Add user_id and log_id placeholders to synthetic data
synthetic_clicks['user_id'] = -1  # Placeholder for synthetic users
synthetic_clicks['log_id'] = range(len(train_processed) + 1, 
                                     len(train_processed) + len(synthetic_clicks) + 1)

print(f"Real no-clicks: {len(real_no_clicks):,}")
print(f"Real clicks: {len(real_clicks):,}")
print(f"Synthetic clicks: {len(synthetic_clicks):,}")

In [ ]:
# Ensure all dataframes have the same columns in the same order
print("Aligning columns...")

# Get all columns from real data
all_columns = real_clicks.columns.tolist()

# Reorder synthetic clicks to match
synthetic_clicks = synthetic_clicks[all_columns]
real_no_clicks = real_no_clicks[all_columns]

print(f"All dataframes aligned with {len(all_columns)} columns")

# Combine all data
print("\nCombining data...")
combined_data = pd.concat([
    real_no_clicks,
    real_clicks,
    synthetic_clicks
], ignore_index=True)

# Shuffle combined dataset
combined_data = combined_data.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"\nCombined data shape: {combined_data.shape}")

# Verify class distribution
print("\n" + "=" * 80)
print("FINAL CLASS DISTRIBUTION")
print("=" * 80)
class_dist_final = combined_data['label'].value_counts().sort_index()
print(f"\nNo-clicks (0): {class_dist_final[0]:,} ({class_dist_final[0]/len(combined_data)*100:.2f}%)")
print(f"Clicks (1): {class_dist_final[1]:,} ({class_dist_final[1]/len(combined_data)*100:.2f}%)")
print(f"\nTotal samples: {len(combined_data):,}")

# Check synthetic data proportion
synthetic_dist = combined_data['is_synthetic'].value_counts()
print(f"\nReal samples: {synthetic_dist[0]:,} ({synthetic_dist[0]/len(combined_data)*100:.2f}%)")
print(f"Synthetic samples: {synthetic_dist[1]:,} ({synthetic_dist[1]/len(combined_data)*100:.2f}%)")

# Visualize final distribution
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
class_dist_final.plot(kind='bar', ax=ax[0], color=['#2E86AB', '#A23B72'])
ax[0].set_title('Final Class Distribution', fontsize=14, fontweight='bold')
ax[0].set_xlabel('Label (0 = No Click, 1 = Click)')
ax[0].set_ylabel('Count')
ax[0].set_xticklabels(['No Click', 'Click'], rotation=0)

# Real vs Synthetic
synthetic_dist.plot(kind='bar', ax=ax[1], color=['#2E86AB', '#F18F01'])
ax[1].set_title('Real vs Synthetic Data', fontsize=14, fontweight='bold')
ax[1].set_xlabel('Data Type')
ax[1].set_ylabel('Count')
ax[1].set_xticklabels(['Real', 'Synthetic'], rotation=0)

plt.tight_layout()
plt.show()

print("\n✓ Data combination complete!")

## 11. Apply Standard Encoding Pipeline

Apply the same encoding used in existing pipeline to the combined dataset:
- One-hot encode low-cardinality categoricals
- Target encode remaining medium-cardinality features  
- Keep numeric features as-is
- Create train/validation split

In [ ]:
# Split into train and validation sets
print("Splitting data into train and validation sets...\n")

train_combined, val_combined = train_test_split(
    combined_data,
    test_size=0.2,
    stratify=combined_data['label'],
    random_state=SEED
)

print(f"Training set: {len(train_combined):,} samples")
print(f"Validation set: {len(val_combined):,} samples")

print(f"\nTraining class distribution:")
print(train_combined['label'].value_counts(normalize=True))

print(f"\nValidation class distribution:")
print(val_combined['label'].value_counts(normalize=True))

In [ ]:
# Define encoding functions
def ohe_fit_transform(train_df, val_df, col, drop_first=True):
    """One-hot encode a categorical column"""
    dtrain = pd.get_dummies(train_df[col], prefix=col, drop_first=drop_first)
    dval = pd.get_dummies(val_df[col], prefix=col, drop_first=drop_first)
    
    # Align columns
    dval = dval.reindex(columns=dtrain.columns, fill_value=0)
    
    # Concatenate back
    train_out = pd.concat([train_df.drop(columns=[col]), dtrain], axis=1)
    val_out = pd.concat([val_df.drop(columns=[col]), dval], axis=1)
    
    return train_out, val_out

def target_encode_fit_transform(train_df, val_df, col, target='label'):
    """Target encode a high-cardinality categorical column"""
    # Skip if already encoded
    if col.endswith('_encoded'):
        return train_df, val_df
    
    # Compute encoding from training data
    encoding = train_df.groupby(col)[target].mean()
    global_mean = train_df[target].mean()
    
    # Create new encoded column
    encoded_col = f"{col}_te"
    train_df[encoded_col] = train_df[col].map(encoding).fillna(global_mean)
    val_df[encoded_col] = val_df[col].map(encoding).fillna(global_mean)
    
    # Drop original column
    train_df = train_df.drop(columns=[col])
    val_df = val_df.drop(columns=[col])
    
    return train_df, val_df

print("Encoding functions defined")

In [ ]:
# Apply encoding pipeline
print("Applying encoding pipeline...\n")

train_encoded = train_combined.copy()
val_encoded = val_combined.copy()

# One-hot encode low-cardinality categoricals (if not already encoded)
low_card_ohe = ['gender', 'net_type', 'creat_type_cd', 'inter_type_cd', 'series_group']

for col in low_card_ohe:
    if col in train_encoded.columns:
        print(f"One-hot encoding: {col}")
        train_encoded, val_encoded = ohe_fit_transform(
            train_encoded, val_encoded, col, drop_first=True
        )

# Target encode remaining medium-cardinality features (if not already encoded)
medium_card_te = [
    'residence', 'series_dev', 'emui_dev', 'hispace_app_tags',
    'app_second_class', 'spread_app_id', 'slot_id', 'device_name',
    'adv_prim_id', 'city'
]

for col in medium_card_te:
    if col in train_encoded.columns and not col.endswith('_encoded'):
        print(f"Target encoding: {col}")
        train_encoded, val_encoded = target_encode_fit_transform(
            train_encoded, val_encoded, col, target='label'
        )

# Drop constant features if present
if 'site_id' in train_encoded.columns:
    train_encoded = train_encoded.drop(columns=['site_id'])
    val_encoded = val_encoded.drop(columns=['site_id'])
    print("Dropped constant feature: site_id")

print(f"\nEncoding complete!")
print(f"Train encoded shape: {train_encoded.shape}")
print(f"Validation encoded shape: {val_encoded.shape}")

In [ ]:
# Prepare data for modeling
print("Preparing data for modeling...\n")

# Separate features and target
X_train = train_encoded.drop(columns=['user_id', 'log_id', 'label', 'is_synthetic'])
y_train = train_encoded['label']

X_val = val_encoded.drop(columns=['user_id', 'log_id', 'label', 'is_synthetic'])
y_val = val_encoded['label']

print(f"Training features shape: {X_train.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Validation features shape: {X_val.shape}")
print(f"Validation target shape: {y_val.shape}")

# Check for any remaining missing values
train_na = X_train.isna().sum().sum()
val_na = X_val.isna().sum().sum()

if train_na > 0 or val_na > 0:
    print(f"\nWARNING: Found {train_na} missing values in training, {val_na} in validation")
    # Fill with 0 if any missing
    X_train = X_train.fillna(0)
    X_val = X_val.fillna(0)
    print("Filled missing values with 0")
else:
    print("\n✓ No missing values found")

## 12. Feature Scaling

Standardize features using StandardScaler (fit on training data only)

In [ ]:
# Scale features
print("Scaling features...\n")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Convert back to DataFrames
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns, index=X_val.index)

print("Features scaled successfully")
print(f"Scaled training shape: {X_train_scaled.shape}")
print(f"Scaled validation shape: {X_val_scaled.shape}")

## 13. Train Logistic Regression

Train logistic regression on the CTGAN-augmented dataset

In [ ]:
# Train logistic regression
print("Training Logistic Regression...\n")

lr_ctgan = LogisticRegression(
    random_state=SEED,
    max_iter=1000,
    n_jobs=-1
)

lr_ctgan.fit(X_train_scaled, y_train)

print("Model trained successfully!")
print(f"Number of features: {len(lr_ctgan.coef_[0])}")

## 14. Evaluate Model Performance

Evaluate the CTGAN-augmented model and compare against baseline models

In [ ]:
# Make predictions
y_pred = lr_ctgan.predict(X_val_scaled)
y_proba = lr_ctgan.predict_proba(X_val_scaled)[:, 1]

# Calculate metrics
results_ctgan = pd.DataFrame({
    'Model': ['LR (CTGAN Synthetic)'],
    'Accuracy': [accuracy_score(y_val, y_pred)],
    'Precision': [precision_score(y_val, y_pred, zero_division=0)],
    'Recall': [recall_score(y_val, y_pred)],
    'F1': [f1_score(y_val, y_pred)],
    'AUC': [roc_auc_score(y_val, y_proba)],
    'Log Loss': [log_loss(y_val, y_proba)]
})

print("=" * 80)
print("MODEL PERFORMANCE (CTGAN SYNTHETIC DATA)")
print("=" * 80)
print("\n")
display(results_ctgan.round(4))

print("\n" + "=" * 80)
print("CONFUSION MATRIX")
print("=" * 80)
cm = confusion_matrix(y_val, y_pred)
print(f"\n{cm}\n")
print(f"True Negatives: {cm[0,0]:,}")
print(f"False Positives: {cm[0,1]:,}")
print(f"False Negatives: {cm[1,0]:,}")
print(f"True Positives: {cm[1,1]:,}")

In [ ]:
# Plot ROC and Precision-Recall curves
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
RocCurveDisplay.from_estimator(
    lr_ctgan, X_val_scaled, y_val, 
    name="LR (CTGAN)", color="#A23B72", ax=ax[0]
)
ax[0].plot([0, 1], [0, 1], 'k--', label='Chance Level')
ax[0].set_title('ROC Curve', fontsize=14, fontweight='bold')
ax[0].legend()

# Precision-Recall Curve
PrecisionRecallDisplay.from_estimator(
    lr_ctgan, X_val_scaled, y_val,
    name="LR (CTGAN)", color="#A23B72", ax=ax[1]
)
pos_rate = y_val.mean()
ax[1].hlines(pos_rate, 0, 1, colors='k', linestyles='--', 
             label=f'No-skill (pos rate={pos_rate:.3f})')
ax[1].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
ax[1].legend()

plt.tight_layout()
plt.show()

## 15. Compare Against Baseline Models

**Baseline Models** (from `logistic_regression_analysis.ipynb`):
1. **Downsampled**: 590K samples, 84.7%/15.3% ratio
2. **Class-weighted**: 6.14M samples, all data with balanced class weights

**New Model**: CTGAN synthetic data, ~7M samples, 85-15 ratio

In [ ]:
# Create comparison table with baseline results
# These values are from logistic_regression_analysis.ipynb

baseline_results = pd.DataFrame({
    'Model': [
        'LR (Downsampled)',
        'LR (Class Weighted)',
        'LR (CTGAN Synthetic)'
    ],
    'Training Samples': [
        '590K',
        '6.14M',
        f'{len(train_combined):,}'
    ],
    'Accuracy': [
        0.9765,
        0.6419,
        results_ctgan['Accuracy'].values[0]
    ],
    'Precision': [
        0.1216,
        0.0306,
        results_ctgan['Precision'].values[0]
    ],
    'Recall': [
        0.0822,
        0.7194,
        results_ctgan['Recall'].values[0]
    ],
    'F1': [
        0.0981,
        0.0587,
        results_ctgan['F1'].values[0]
    ],
    'AUC': [
        0.7490,
        0.7499,
        results_ctgan['AUC'].values[0]
    ]
})

print("=" * 100)
print("MODEL COMPARISON: Baseline vs CTGAN Synthetic Data")
print("=" * 100)
print("\n")
display(baseline_results.round(4))

print("\n" + "=" * 100)
print("KEY INSIGHTS")
print("=" * 100)
print("""
1. **Downsampled Model**:
   - Uses only 590K samples (10% of data)
   - High accuracy (97.65%) but low recall (8.22%)
   - Misses most positive cases

2. **Class-Weighted Model**:
   - Uses all 6.14M samples with balanced weights
   - High recall (71.94%) but low precision (3.06%)
   - Many false positives

3. **CTGAN Synthetic Model** (This notebook):
   - Uses real + synthetic data for 85-15 ratio
   - Balanced approach between precision and recall
   - Leverages synthetic data to address class imbalance
""")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
colors = ['#2E86AB', '#F18F01', '#A23B72']

for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    
    values = baseline_results[metric].values
    models = baseline_results['Model'].values
    
    bars = ax.bar(models, values, color=colors)
    ax.set_title(f'{metric} Comparison', fontsize=12, fontweight='bold')
    ax.set_ylabel(metric)
    ax.set_ylim(0, max(values) * 1.2)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}',
                ha='center', va='bottom', fontsize=10)
    
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## 16. Save Results and Models

Save trained models, results, and datasets for future use

In [ ]:
# Save results
print("Saving results...\n")

# Save trained model
with open('lr_ctgan_model.pkl', 'wb') as f:
    pickle.dump(lr_ctgan, f)
print("✓ Saved trained LR model: lr_ctgan_model.pkl")

# Save scaler
with open('ctgan_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✓ Saved scaler: ctgan_scaler.pkl")

# Save combined dataset (optional - can be large)
# with open('combined_data_ctgan.pkl', 'wb') as f:
#     pickle.dump(combined_data, f)
# print("✓ Saved combined dataset: combined_data_ctgan.pkl")

# Save results comparison
baseline_results.to_csv('model_comparison_ctgan.csv', index=False)
print("✓ Saved comparison results: model_comparison_ctgan.csv")

# Save encoding maps
with open('encoding_maps.pkl', 'wb') as f:
    pickle.dump({
        'target_encoding': encoding_maps,
        'grouping': grouping_maps
    }, f)
print("✓ Saved encoding maps: encoding_maps.pkl")

print("\nAll results saved successfully!")

## 17. Conclusions and Next Steps

### Summary

This notebook implemented a CTGAN-based approach to address class imbalance in CTR prediction:

1. **Loaded and merged** raw data from CSV files
2. **Applied hybrid preprocessing**:
   - Target encoded very high-cardinality features (1000+ unique)
   - Grouped rare categories for medium-high cardinality features (100-1000 unique)
   - Kept low-cardinality features as categorical
3. **Trained CTGAN** on minority class (clicks only)
4. **Generated synthetic clicks** to achieve 85-15 ratio
5. **Combined real + synthetic data** and applied standard encoding pipeline
6. **Trained and evaluated** logistic regression
7. **Compared** against baseline models

### Key Findings

- CTGAN successfully generated synthetic click examples
- Synthetic data distributions match real data reasonably well
- Model performance compared to baseline approaches
- Achieved better balance between precision and recall

### Next Steps

1. **Hyperparameter tuning**: Optimize CTGAN parameters (epochs, batch size, learning rates)
2. **Quality validation**: Statistical tests to validate synthetic data quality
3. **Alternative ratios**: Experiment with different class ratios (e.g., 90-10, 80-20)
4. **Feature importance**: Analyze which features contribute most to predictions
5. **Other models**: Test synthetic data with other algorithms (Random Forest, XGBoost)
6. **Production deployment**: Integrate best model into production pipeline